# Predicting particle orientation from position and aspect ratio

### B.Tech project, Department of Chemical Engineering, IIT Roorkee (2024–25)

For elliptical-cylinder particles in a vibrated bed simulated with the Discrete Element Method (DEM), I predict each particle's orientation angle (0–180°) from its position `(x, y, z)` and its `aspect_ratio` using a Random Forest.

The data are particle-level outputs of the DEM simulations, one CSV file per simulation run, with about 3.09 million rows in total. The files are too large for this repository, so the notebook is shared with its saved outputs. To rerun it, place the CSV files in `data/`.

**Workflow:** load and combine the files, clean them, split the rows 80/20 at random, fit a Random Forest, and evaluate it on the held-out 20%.

In [1]:
import pandas as pd
import os


## 1. Load the data
I read the five columns I need from every CSV file and stack them into one table.

In [2]:
import glob

# Folder with the DEM particle CSV files
folder_path = '../data'

# Columns used in the analysis
selected_columns = ['Points:0', 'Points:1', 'Points:2', 'aspect_ratio', 'Orientation']

In [3]:
# Use glob to find all CSV files in the directory
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Combine all CSV files while selecting only the required columns
combined_df = pd.concat([pd.read_csv(file, usecols=selected_columns) for file in all_files], ignore_index=True)

# Check the combined DataFrame
print(combined_df.head())
print("Total Rows:", len(combined_df))


   Points:0  Points:1  Points:2 aspect_ratio  Orientation
0  0.018007  0.002985  0.002993         0.75    67.914578
1  0.023991  0.002991  0.002991         0.75   101.649383
2  0.029984  0.003016  0.002999         0.75    44.223714
3  0.035975  0.002998  0.002994         0.75    38.687792
4   0.04204  0.002994  0.002996         0.75    50.187793
Total Rows: 3090008


In [24]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3090008 entries, 0 to 3090007
Data columns (total 5 columns):
 #   Column        Dtype  
---  ------        -----  
 0   Points:0      object 
 1   Points:1      object 
 2   Points:2      object 
 3   aspect_ratio  object 
 4   Orientation   float64
dtypes: float64(1), object(4)
memory usage: 117.9+ MB


In [25]:
print(combined_df.isna().sum())

Points:0        7426
Points:1        7426
Points:2        7426
aspect_ratio       0
Orientation       17
dtype: int64


## 2. Clean the data
Some coordinate cells are empty or non-numeric: 7,426 rows lack coordinates and 17 lack an orientation value. I convert every column to numeric and drop the rows with missing values, which removes 7,443 rows and leaves 3,082,565.

In [26]:
combined_df = combined_df.apply(pd.to_numeric, errors='coerce')
combined_df.dropna(inplace=True)
print(combined_df.isna().sum())

Points:0        0
Points:1        0
Points:2        0
aspect_ratio    0
Orientation     0
dtype: int64


In [27]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3082565 entries, 0 to 3090007
Data columns (total 5 columns):
 #   Column        Dtype  
---  ------        -----  
 0   Points:0      float64
 1   Points:1      float64
 2   Points:2      float64
 3   aspect_ratio  float64
 4   Orientation   float64
dtypes: float64(5)
memory usage: 141.1 MB


## 3. Model
The inputs are the three coordinates and the aspect ratio; the target is the orientation angle. I split the rows 80/20 at random (`random_state=10`) and fit a Random Forest with 50 trees.

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [30]:
X=combined_df[['Points:0','Points:1','Points:2','aspect_ratio']]
y=combined_df['Orientation']
x_train, x_test, y_train, y_test = train_test_split(X,y,random_state=10,test_size=0.2)

In [50]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=50,
                                  random_state=0)

In [51]:
model.fit(x_train,y_train)

RandomForestRegressor(n_estimators=50, random_state=0)

In [52]:
y_pred = model.predict(x_test)

## 4. Results
I evaluate on the 20% test set (616,513 particles) with two measures.

* **R²**: the standard coefficient of determination.
* **Within-tolerance accuracy**: the share of predictions whose relative error `|y − ŷ| / y` is at most 35%.

The Random Forest reaches an **R² of 0.69**, and **83.3%** of its predictions fall within 35% of the true angle.

In [53]:
score_R2=r2_score(y_test,y_pred)
score_R2

0.6906511097868053

In [56]:
import numpy as np

def within_tolerance_accuracy(y_true, y_pred, tolerance=0.35):
    # Share of predictions whose relative error |y - y_hat| / y is within the tolerance
    relative_error = np.abs((y_true - y_pred) / y_true)
    return np.sum(relative_error <= tolerance) / len(y_true)

score_within_tolerance = within_tolerance_accuracy(y_test, y_pred, tolerance=0.35)
score_within_tolerance

0.8333271155677171

## 5. Notes and Limitations
* Both figures come from a single random split of all particle rows pooled together. Particles from the same simulation resemble each other, so these scores describe interpolation within the simulated runs more than transfer to a new simulation. Testing on held-out simulation runs is the natural next step.
* Other extensions I would try: held-out aspect ratios and held-out stages as separate tests, and local features such as neighbour count and local density alongside position.